<a href="https://colab.research.google.com/github/BarisEmreRuzgar/Adli-Fotorobot/blob/main/Adli_Fotorobot_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

packages = [
    "torch==2.1.0+cu121 torchvision==0.16.0+cu121 --index-url https://download.pytorch.org/whl/cu121",
    "diffusers==0.27.2",
    "transformers==4.40.0",
    "accelerate==0.30.0",
    "huggingface_hub==0.23.0",
    "safetensors==0.4.3",
    "Pillow==10.3.0",
    "opencv-python==4.9.0.80",
    "insightface==0.7.3",
    "onnxruntime-gpu==1.18.0",
    "google-genai==1.5.0",
    "gradio==4.44.0",
    "peft==0.11.0",
    "xformers==0.0.25.post1",
]

for pkg in packages:
    subprocess.run(
        f"{sys.executable} -m pip install -q {pkg}",
        shell=True, check=False
    )

subprocess.run(
    "pip install -q git+https://github.com/tencent-ailab/IP-Adapter.git",
    shell=True, check=False
)

print("✅ Kurulum tamam!")
print("⚠️  Runtime → Restart and Run All yapın!")

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("/content/models/ip_adapter", exist_ok=True)

files = [
    (
        "h94/IP-Adapter-FaceID",
        "ip-adapter-faceid-plusv2_sd15.bin",
        "/content/models/ip_adapter"
    ),
]

for repo, filename, folder in files:
    out = os.path.join(folder, filename)
    if not os.path.exists(out):
        print(f"⏳ İndiriliyor: {filename}")
        hf_hub_download(repo_id=repo, filename=filename, local_dir=folder)
        print(f"✅ {filename} indirildi")
    else:
        print(f"✅ Zaten var: {filename}")

print("\n✅ Modeller hazır!")

In [ ]:
import os, json, re, io, cv2, torch
import numpy as np
import gradio as gr
from PIL import Image
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    DPMSolverMultistepScheduler,
    AutoencoderKL,
)
from insightface.app import FaceAnalysis
from ip_adapter.ip_adapter_faceid import IPAdapterFaceIDPlus
from google import genai
from google.genai import types

# ── CONFIG ───────────────────────────────────────────────────────
GOOGLE_API_KEY = "BURAYA_GEMINI_API_KEY"
GEMINI_MODEL   = "gemini-2.5-flash"
SD_MODEL       = "SG161222/Realistic_Vision_V5.1_noVAE"
SD_VAE         = "stabilityai/sd-vae-ft-mse"
IMAGE_ENCODER  = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
IP_CKPT        = "/content/models/ip_adapter/ip-adapter-faceid-plusv2_sd15.bin"
DEVICE         = "cuda"

# ── GEMİNİ ───────────────────────────────────────────────────────
print("🔗 Gemini bağlanıyor...")
client = genai.Client(api_key=GOOGLE_API_KEY)
try:
    test = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=["OK"]
    )
    print("✅ Gemini OK")
except Exception as e:
    print(f"❌ Gemini hata: {e}")

# ── FACE ANALYSIS ─────────────────────────────────────────────────
print("⏳ InsightFace yükleniyor...")
face_app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print("✅ InsightFace hazır!")

# ── VAE ──────────────────────────────────────────────────────────
print("⏳ VAE yükleniyor...")
vae = AutoencoderKL.from_pretrained(
    SD_VAE,
    torch_dtype=torch.float16
).to(DEVICE)
print("✅ VAE hazır!")

# ── TXT2IMG PIPELINE ─────────────────────────────────────────────
print("⏳ txt2img pipeline yükleniyor...")
pipe_txt = StableDiffusionPipeline.from_pretrained(
    SD_MODEL,
    vae=vae,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to(DEVICE)
pipe_txt.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe_txt.scheduler.config,
    algorithm_type="dpmsolver++",
    use_karras_sigmas=True,
)
try:
    pipe_txt.enable_xformers_memory_efficient_attention()
    print("✅ xformers aktif (txt2img)")
except Exception:
    pipe_txt.enable_attention_slicing()
    print("⚠️ xformers yok, attention slicing kullanılıyor (txt2img)")
print("✅ txt2img hazır!")

# ── IMG2IMG PIPELINE ─────────────────────────────────────────────
print("⏳ img2img pipeline yükleniyor...")
pipe_i2i = StableDiffusionImg2ImgPipeline(
    vae=pipe_txt.vae,
    text_encoder=pipe_txt.text_encoder,
    tokenizer=pipe_txt.tokenizer,
    unet=pipe_txt.unet,
    scheduler=pipe_txt.scheduler,
    safety_checker=None,
    feature_extractor=None,
    requires_safety_checker=False,
).to(DEVICE)
try:
    pipe_i2i.enable_xformers_memory_efficient_attention()
    print("✅ xformers aktif (img2img)")
except Exception:
    pipe_i2i.enable_attention_slicing()
    print("⚠️ xformers yok, attention slicing kullanılıyor (img2img)")
print("✅ img2img hazır!")

# ── IP-ADAPTER ────────────────────────────────────────────────────
print("⏳ IP-Adapter FaceID yükleniyor...")
ip_model = IPAdapterFaceIDPlus(
    sd_pipe=pipe_i2i,
    image_encoder_path=IMAGE_ENCODER,
    ip_ckpt=IP_CKPT,
    lora_rank=128,
    num_tokens=4,
    device=DEVICE,
)
print("✅ IP-Adapter hazır!")
print("\n🎉 Tüm modeller yüklendi!")

# ── YARDIMCI FONKSİYONLAR ────────────────────────────────────────
def numpy_to_pil(img):
    if img is None:
        return None
    if isinstance(img, np.ndarray):
        return Image.fromarray(img).convert("RGB")
    return img.convert("RGB")

def pil_to_cv2(pil_img):
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def gorsel_hazirla(pil_img, boyut=512):
    img    = pil_img.convert("RGB")
    w, h   = img.size
    oran   = min(boyut / w, boyut / h)
    yeni_w = int(w * oran)
    yeni_h = int(h * oran)
    img    = img.resize((yeni_w, yeni_h), Image.LANCZOS)
    zemin  = Image.new("RGB", (boyut, boyut), (180, 175, 170))
    x = (boyut - yeni_w) // 2
    y = (boyut - yeni_h) // 2
    zemin.paste(img, (x, y))
    return zemin

def yuz_embedding_al(pil_img):
    cv2_img = pil_to_cv2(pil_img)
    yuzler  = face_app.get(cv2_img)
    if not yuzler:
        return None, "❌ Görselde yüz bulunamadı!"
    yuz = sorted(
        yuzler,
        key=lambda x: (x.bbox[2] - x.bbox[0]) * (x.bbox[3] - x.bbox[1]),
        reverse=True
    )[0]
    # ✅ IPAdapterFaceIDPlus için doğru shape: [1, 512]
    emb = torch.from_numpy(yuz.normed_embedding).unsqueeze(0)
    return emb, "✅ Yüz embedding alındı"

def gorsel_sec(evt: gr.SelectData, galeri):
    try:
        if not galeri:
            return None
        item = galeri[evt.index]
        if isinstance(item, (list, tuple)):
            path = item[0]
        elif isinstance(item, dict):
            path = item.get("image", item.get("name"))
        else:
            path = item
        if isinstance(path, str):
            return Image.open(path).convert("RGB")
        return path
    except Exception as e:
        print(f"gorsel_sec hata: {e}")
        return None

def gorseli_aktar(gorsel, params):
    if gorsel is None:
        return None, params
    if isinstance(gorsel, np.ndarray):
        return Image.fromarray(gorsel).convert("RGB"), params
    return gorsel, params

# ── AŞAMA 1 ───────────────────────────────────────────────────────
def birinci_asama_uret(tanik_metni):
    rapor = []
    if not tanik_metni.strip():
        return [], {}, "❌ Tanık ifadesi boş!"

    talimat = """Sen bir adli fotorobot asistanisin.
Asagidaki JSON formatini KESINLIKLE doldur. Tum degerler INGILIZCE olmali.
Baska hicbir sey yazma, sadece JSON:
{
  "yas": "20-year-old",
  "cinsiyet": "male",
  "yuz_sekli": "oval face",
  "goz": "dark brown almond-shaped eyes",
  "burun": "straight medium nose",
  "dudak": "defined cupid bow lips",
  "cene": "strong V-shaped jawline",
  "sac": "short dark brown taper fade hair",
  "sakal": "clean shaven",
  "ten": "warm olive skin tone",
  "ozel": ""
}"""

    try:
        resp   = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[talimat + "\n\nTanik ifadesi:\n" + tanik_metni]
        )
        raw    = re.sub(r"```json|```", "", resp.text.strip()).strip()
        params = json.loads(raw)
        rapor.append("✅ Gemini analiz OK")
    except Exception as e:
        rapor.append(f"⚠️ Gemini hata ({e}), varsayılan kullanılıyor")
        params = {
            "yas": "20-year-old", "cinsiyet": "male",
            "yuz_sekli": "oval face", "goz": "dark brown almond eyes",
            "burun": "straight nose", "dudak": "defined lips",
            "cene": "strong jawline", "sac": "short dark brown hair",
            "sakal": "clean shaven", "ten": "warm olive skin", "ozel": ""
        }

    for k, v in params.items():
        rapor.append(f"  {k:<10}: {v}")

    ozel   = f", {params['ozel']}" if params.get("ozel") else ""
    prompt = (
        f"(RAW photo, best quality, masterpiece:1.3), "
        f"1man, solo, {params['yas']}, {params['yuz_sekli']}, "
        f"{params['ten']}, {params['sac']}, {params['sakal']}, "
        f"{params['goz']}, {params['burun']}, {params['dudak']}, "
        f"{params['cene']}{ozel}, "
        f"looking straight at camera, neutral expression, "
        f"plain light grey background, upper body, wearing shirt, "
        f"professional studio lighting, sharp focus, photorealistic, "
        f"(8k uhd:1.1), natural skin texture, skin pores visible"
    )
    negative = (
        "(worst quality, low quality:1.4), lowres, blurry, "
        "jpeg artifacts, cartoon, anime, painting, illustration, "
        "deformed, bad anatomy, disfigured, extra limbs, mutated, "
        "female, woman, girl, feminine, "
        "hat, glasses, watermark, text, logo, nsfw"
    )

    sonuclar = []
    for i in range(4):
        try:
            gen = torch.Generator(DEVICE).manual_seed(i * 1337 + 42)
            out = pipe_txt(
                prompt=prompt,
                negative_prompt=negative,
                width=512, height=512,
                guidance_scale=7.0,
                num_inference_steps=45,
                generator=gen,
            )
            sonuclar.append(out.images[0])
            rapor.append(f"  Varyasyon {i+1}/4 ✔")
        except Exception as e:
            rapor.append(f"  Varyasyon {i+1} HATA: {e}")

    return sonuclar, params, "\n".join(rapor)

# ── AŞAMA 2 ───────────────────────────────────────────────────────
def ikinci_asama_evrimlestir(evrim_metni, secilen_gorsel, mevcut_params):
    rapor = []

    if secilen_gorsel is None:
        return [], mevcut_params, "❌ Önce Tab 1'den bir yüz seçin!"
    if not evrim_metni or not evrim_metni.strip():
        return [], mevcut_params, "❌ Değişiklik metni boş!"

    init_pil = numpy_to_pil(secilen_gorsel)
    init_pil = gorsel_hazirla(init_pil, 512)

    face_emb, emb_msg = yuz_embedding_al(init_pil)
    rapor.append(emb_msg)
    if face_emb is None:
        return [], mevcut_params, "\n".join(rapor)

    try:
        buf = io.BytesIO()
        init_pil.save(buf, format="JPEG", quality=92)
        img_bytes = buf.getvalue()

        vision_prompt = f"""Analyze this male portrait photo carefully.
Identify ALL current facial features: exact age range, face shape, skin tone,
hair style and color, eye color and shape, nose shape, lip shape, jawline, beard status.

User wants ONLY these changes: "{evrim_metni}"

Return a SINGLE LINE Stable Diffusion prompt in English.
- Keep ALL other features exactly as they are
- Only change what was requested
- Must start with: 1man, solo,
- Include all features: age, face shape, skin, hair, eyes, nose, lips, jaw, beard
- Max 60 words, plain text only, no JSON, no explanation"""

        resp = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
                vision_prompt,
            ]
        )
        yeni_prompt = resp.text.strip().strip('"\'')
        yeni_prompt = re.sub(r"```.*?```", "", yeni_prompt, flags=re.DOTALL).strip()
        rapor.append(f"✅ Vision prompt: {yeni_prompt}")

    except Exception as e:
        rapor.append(f"⚠️ Vision hata ({e}), fallback kullanılıyor")
        p = mevcut_params or {}
        yeni_prompt = (
            f"1man, solo, {p.get('yas','young man')}, "
            f"{p.get('yuz_sekli','oval face')}, {p.get('ten','olive skin')}, "
            f"{p.get('goz','brown eyes')}, {p.get('burun','straight nose')}, "
            f"{p.get('cene','strong jaw')}, {p.get('sac','short hair')}, "
            f"{p.get('sakal','clean shaven')}"
        )

    tam_prompt = (
        f"(RAW photo, best quality, masterpiece:1.3), "
        f"{yeni_prompt}, "
        f"looking straight at camera, neutral expression, "
        f"plain light grey background, upper body, wearing shirt, "
        f"professional studio lighting, sharp focus, photorealistic, "
        f"(8k uhd:1.1), natural skin texture"
    )
    negative = (
        "(worst quality, low quality:1.4), lowres, blurry, "
        "deformed, bad anatomy, disfigured, "
        "cartoon, anime, painting, female, woman, girl, "
        "different person, identity change, "
        "hat, glasses, watermark, text, nsfw"
    )

    sonuclar = []
    seeds = [300, 317, 334]

    for i, seed in enumerate(seeds):
        try:
            gen = torch.Generator(DEVICE).manual_seed(seed)
            out = ip_model.generate(
                prompt=tam_prompt,
                negative_prompt=negative,
                faceid_embeds=face_emb,
                num_samples=1,
                width=512,
                height=512,
                guidance_scale=8.0,
                num_inference_steps=50,
                generator=gen,
            )
            sonuclar.append(out[0])
            rapor.append(f"  Varyasyon {i+1} (IP-Adapter) ✔")
        except Exception as e:
            rapor.append(f"  Varyasyon {i+1} IP-Adapter HATA: {e}")
            try:
                out2 = pipe_i2i(
                    prompt=tam_prompt,
                    negative_prompt=negative,
                    image=init_pil,
                    strength=0.55,
                    guidance_scale=8.0,
                    num_inference_steps=50,
                    generator=gen,
                )
                sonuclar.append(out2.images[0])
                rapor.append(f"    → Fallback img2img ✔")
            except Exception as e2:
                rapor.append(f"    → Fallback da başarısız: {e2}")

    if not sonuclar:
        return [], mevcut_params, "❌ Hiç görsel üretilemedi!\n" + "\n".join(rapor)

    return sonuclar, mevcut_params, "\n".join(rapor)

# ── GRADIO UI ─────────────────────────────────────────────────────
with gr.Blocks(
    title="🔍 Adli Fotorobot V3.0",
    theme=gr.themes.Soft(),
    css=".gradio-container {max-width: 1400px !important}"
) as demo:

    gr.Markdown("""
# 🔍 Adli Fotorobot V3.0
### Gemini AI + Realistic Vision V5.1 + IP-Adapter FaceID Plus V2
    """)

    params_state = gr.State({})

    with gr.Tabs():

        with gr.TabItem("1️⃣  İlk Jenerasyon"):
            with gr.Row():
                with gr.Column(scale=1):
                    tanik_ifadesi = gr.Textbox(
                        label="📝 Tanık İfadesi",
                        lines=14,
                        placeholder=(
                            "19-20 yaşında bir erkek. Kısa koyu kahverengi saçlı, "
                            "temiz tıraşlı, oval yüzlü, iri koyu kahverengi gözlü..."
                        )
                    )
                    btn_uret = gr.Button(
                        "🎨 Yüz Üret (4 Varyasyon)",
                        variant="primary", size="lg"
                    )
                    rapor_s1 = gr.Textbox(
                        label="📋 Analiz Raporu",
                        lines=12, interactive=False
                    )

                with gr.Column(scale=2):
                    galeri_s1 = gr.Gallery(
                        label="🖼️ Üretilen Yüzler — Birini Seç",
                        columns=2, rows=2,
                        height=520, object_fit="cover"
                    )
                    secilen_s1 = gr.Image(
                        label="✅ Seçilen Şüpheli",
                        height=260, interactive=False
                    )
                    btn_gonder = gr.Button(
                        "➡️ Bu Yüzü Aşama 2'ye Gönder",
                        variant="secondary", size="lg"
                    )

            galeri_s1.select(
                fn=gorsel_sec,
                inputs=[galeri_s1],
                outputs=[secilen_s1]
            )
            btn_uret.click(
                fn=birinci_asama_uret,
                inputs=[tanik_ifadesi],
                outputs=[galeri_s1, params_state, rapor_s1]
            )

        with gr.TabItem("2️⃣  Özellik Güncelleme (IP-Adapter)"):
            gr.Markdown("""
### 💡 IP-Adapter FaceID Plus V2 — Kimlik korunarak özellik değiştirme
Her değişikliği **ayrı satıra** yaz.

| Kategori | Örnek |
|----------|-------|
| **Saç** | `saçları kısa buzz cut yap` |
| **Sakal** | `sakalı temizle` / `kısa kare sakal ekle` |
| **Göz** | `gözler mavi olsun` |
| **Ten** | `ten rengi açık olsun` |
| **Yaş** | `daha yaşlı göster` |
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    secilen_s2 = gr.Image(
                        label="🖼️ Düzenlenecek Yüz",
                        height=360, interactive=False
                    )
                    degisiklik = gr.Textbox(
                        label="✏️ Ne Değişsin?",
                        lines=6,
                        placeholder="saçları kısa buzz cut yap\nsakalı temizle"
                    )
                    btn_degistir = gr.Button(
                        "🔄 Güncelle",
                        variant="primary", size="lg"
                    )
                    rapor_s2 = gr.Textbox(
                        label="📋 İşlem Raporu",
                        lines=7, interactive=False
                    )

                with gr.Column(scale=2):
                    galeri_s2 = gr.Gallery(
                        label="🖼️ Güncellenmiş Yüzler (3 Varyasyon)",
                        columns=3, rows=1,
                        height=400, object_fit="cover"
                    )
                    secilen_s2_out = gr.Image(
                        label="✅ Seçilen Sonuç",
                        height=300, interactive=False
                    )

            galeri_s2.select(
                fn=gorsel_sec,
                inputs=[galeri_s2],
                outputs=[secilen_s2_out]
            )
            btn_gonder.click(
                fn=gorseli_aktar,
                inputs=[secilen_s1, params_state],
                outputs=[secilen_s2, params_state]
            )
            btn_degistir.click(
                fn=ikinci_asama_evrimlestir,
                inputs=[degisiklik, secilen_s2, params_state],
                outputs=[galeri_s2, params_state, rapor_s2]
            )

demo.launch(share=True, debug=True)